# Evaluasi Knowledge Graph SEPSES CSKG

Notebook ini menjalankan evaluasi lengkap terhadap Knowledge Graph yang dibangun oleh pipeline agentic.

**Author**: Mikail Achmad (Evaluation System)

**Tugas**:
- Koneksi ke SPARQL endpoint Qlever
- Hitung statistik KG (triple, entitas, relasi)
- Evaluasi per sumber data (CVE, CVSS, CWE, CPE, CAPEC, ATT&CK, ICSA)
- Evaluasi kualitas linking
- Identifikasi missing links / error
- Visualisasi (chart + tabel)


In [ ]:
# Setup path supaya bisa import dari src/
import sys
from pathlib import Path

# Pastikan root project ada di path
ROOT = Path().resolve().parent  # notebooks/ → root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")

## 1. Koneksi ke SPARQL Endpoint

In [ ]:
from src.sparql.sparql_client import SparqlClient

# Ganti URL jika endpoint berjalan di port/host berbeda
ENDPOINT_URL = "http://localhost:7001/sparql"

client = SparqlClient(endpoint_url=ENDPOINT_URL)

# Cek apakah endpoint aktif
is_active = client.ping(retries=3)
print(f"Endpoint aktif: {is_active}")

## 2. Jalankan Evaluasi Lengkap

In [ ]:
from src.evaluation.kg_evaluator import KGEvaluator

evaluator = KGEvaluator(client)

# Jalankan semua evaluasi sekaligus
# Jika endpoint belum siap, ganti dengan _demo_stats() (lihat cell berikutnya)
stats = evaluator.run_full_evaluation()

In [ ]:
# [OPSIONAL] Gunakan data dummy jika endpoint belum siap
# Hapus komentar di bawah ini untuk mengaktifkan mode demo

# from src.evaluation.kg_visualizer import _demo_stats
# stats = _demo_stats()
# print("Mode demo aktif - menggunakan data dummy")

## 3. Tampilkan Tabel Statistik

In [ ]:
import pandas as pd

df = stats.to_dataframe()

# Tampilkan per kategori
for cat in df["Kategori"].unique():
    print(f"\n{'='*40}")
    print(f"  {cat}")
    print('='*40)
    subset = df[df["Kategori"] == cat][["Metrik", "Nilai"]]
    subset["Nilai"] = subset["Nilai"].apply(lambda x: f"{x:,}")
    print(subset.to_string(index=False))

In [ ]:
# Tampilkan missing links
print("\nMissing Links (entitas tanpa koneksi penting):")
print("-" * 40)
for k, v in stats.missing_links.items():
    print(f"  {k:30s}: {v:>10,}")

## 4. Visualisasi

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from src.evaluation.kg_visualizer import (
    plot_entities_per_source,
    plot_source_distribution_pie,
    plot_link_quality,
    plot_summary_table,
)

OUTPUT_DIR = ROOT / "docs" / "evaluation"

# Chart 1: Bar chart entitas per sumber
plot_entities_per_source(stats, output_dir=OUTPUT_DIR)
plt.show()

In [ ]:
# Chart 2: Pie chart distribusi sumber
plot_source_distribution_pie(stats, output_dir=OUTPUT_DIR)
plt.show()

In [ ]:
# Chart 3: Grouped bar - kualitas linking
plot_link_quality(stats, output_dir=OUTPUT_DIR)
plt.show()

In [ ]:
# Chart 4: Summary table sebagai gambar
plot_summary_table(stats, output_dir=OUTPUT_DIR)
plt.show()

## 5. Simpan Hasil ke CSV

In [ ]:
csv_path = OUTPUT_DIR / "kg_stats.csv"
evaluator.save_stats_csv(stats, csv_path)
print(f"Statistik disimpan ke: {csv_path}")

## 6. Contoh Query Manual ke Endpoint

In [ ]:
# Contoh: ambil 10 CVE pertama beserta CVSS score-nya
results = client.query("""
    SELECT ?cve ?cvssScore WHERE {
        ?cve a cve:CVE .
        ?cve cvss:hasCVSS ?cvss .
        ?cvss cvss:baseScore ?cvssScore .
    }
    ORDER BY DESC(?cvssScore)
    LIMIT 10
""")

print("Top 10 CVE by CVSS Score:")
for r in results:
    cve_id = r["cve"]["value"].split("/")[-1]
    score  = r["cvssScore"]["value"]
    print(f"  {cve_id:20s}: {score}")